# 💼 WealthWise AI — Portfolio Recommendation Engine
### Notebook 05: Risk-Driven Asset Allocation & Personalised Investment Strategy

---

## 📋 Table of Contents

| Section | Topic |
|---------|-------|
| 1  | Project Introduction |
| 2  | Import Libraries |
| 3  | Load Datasets |
| 4  | Data Validation & Merge |
| 5  | Portfolio Recommendation Engine |
| 6  | Generate Portfolio Allocations |
| 7  | Investment Strategy Generator |
| 8  | Health Score Based Advice |
| 9  | Visualisation |
| 10 | Recommendation Summary |
| 11 | Export Results |
| 12 | Business Interpretation |

---

> **Author:** WealthWise AI Engineering Team  
> **Version:** 1.0.0  
> **Date:** June 2026  
> **Status:** Production-Ready  
> **Inputs:** `health_score_dataset.csv`, `risk_profile_predictions.csv`  
> **Outputs:** `portfolio_recommendation_dataset.csv`, `portfolio_recommendation_dataset.xlsx`

---

# Section 1 — Project Introduction

---

## 🚀 WealthWise AI: Platform Status

| Module | Notebook | Status |
|--------|----------|--------|
| 🧹 Data Cleaning | 01_data_cleaning | ✅ Complete |
| 🔧 Feature Engineering | 02_feature_engineering | ✅ Complete |
| 🤖 Risk Profile Classification | 03_risk_profile_model | ✅ Complete |
| 🩺 Financial Health Score | 04_financial_health_score | ✅ Complete |
| 💼 **Portfolio Recommendation** | **05_portfolio_recommendation** | 🔄 **This Notebook** |
| 🧠 AI Financial Coach | 06_ai_financial_coach | 🔜 Coming Next |

---

## 💼 What is Portfolio Recommendation?

**Portfolio Recommendation** is the process of allocating a user's investable capital across different **asset classes** in proportions that match their:
- **Risk tolerance** — determined by the ML Risk Profile Model (Notebook 03)
- **Financial readiness** — assessed by the Financial Health Score (Notebook 04)

### The Five Asset Classes Used

| Asset Class | Type | Risk Level | Liquidity |
|-------------|------|------------|----------|
| **Stocks** (Equities) | Growth | High | High |
| **Mutual Funds** | Balanced | Medium | High |
| **Gold** | Hedge / Store of value | Low-Medium | Medium |
| **Fixed Deposit (FD)** | Capital preservation | Very Low | Low |
| **Crypto** | Speculative growth | Very High | High |

---

## 🎯 Why Risk-Based Investing?

The fundamental principle of modern portfolio theory (Markowitz, 1952) states that **risk and return are inseparable**. An investor's optimal portfolio is the one that maximises expected return for their accepted level of risk.

| Risk Profile | Investment Philosophy |
|-------------|----------------------|
| 🟢 **Conservative** | Capital preservation first — minimal equity exposure, maximum stable instruments |
| 🟡 **Moderate** | Balanced growth — diversified across growth and stability instruments |
| 🔴 **Aggressive** | Wealth creation — high equity + alternative assets for maximum long-term return |

---

## 📊 WealthWise AI Allocation Rules

| Asset | Conservative | Moderate | Aggressive |
|-------|-------------|----------|------------|
| Stocks | 10% | 30% | 50% |
| Mutual Funds | 30% | 40% | 30% |
| Gold | 20% | 15% | 10% |
| Fixed Deposit | 40% | 15% | 0% |
| Crypto | 0% | 0% | 10% |
| **Total** | **100%** | **100%** | **100%** |

---

## 🔗 Data Flow: How This Notebook Fits

```
features_dataset.csv
       |
       +--------> [Notebook 03] risk_profile_predictions.csv
       |                  predicted_risk_profile  (test-set rows)
       |
       +--------> [Notebook 04] health_score_dataset.csv
                         health_score, health_category  (all rows)
                              |
                              v
               [Notebook 05] MERGE on account_number + month_year
                              |
                              v
            Portfolio Recommendation Engine
                              |
                              v
       portfolio_recommendation_dataset.csv / .xlsx
```

---

# Section 2 — Import Libraries

---

In [1]:
# =============================================================================
# SECTION 2: LIBRARY IMPORTS
# =============================================================================

# Standard library
import os
import warnings

# Data manipulation
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns

# ── Suppress non-critical warnings ──────────────────────────────────────────
warnings.filterwarnings('ignore')

# ── Pandas display settings ───────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 90)
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.width', 160)

# ── Chart theme ───────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
FIGURE_DPI = 120

# ── WealthWise AI Brand Palette ───────────────────────────────────────────────
RISK_COLORS = {
    'Conservative': '#27AE60',   # Emerald green
    'Moderate':     '#F39C12',   # Warm amber
    'Aggressive':   '#C0392B',   # Crimson red
}

HEALTH_COLORS = {
    'Excellent': '#1ABC9C',
    'Good':      '#3498DB',
    'Fair':      '#E67E22',
    'Poor':      '#95A5A6',
}

ASSET_COLORS = {
    'Stocks':       '#E74C3C',
    'Mutual Funds': '#3498DB',
    'Gold':         '#F1C40F',
    'Fixed Deposit':'#27AE60',
    'Crypto':       '#9B59B6',
}

# ── Portfolio allocation rules — single source of truth ──────────────────────
# Keys match asset names used throughout this notebook.
PORTFOLIO_RULES = {
    'Conservative': {
        'stocks_pct':       10,
        'mutual_funds_pct': 30,
        'gold_pct':         20,
        'fd_pct':           40,
        'crypto_pct':        0,
    },
    'Moderate': {
        'stocks_pct':       30,
        'mutual_funds_pct': 40,
        'gold_pct':         15,
        'fd_pct':           15,
        'crypto_pct':        0,
    },
    'Aggressive': {
        'stocks_pct':       50,
        'mutual_funds_pct': 30,
        'gold_pct':         10,
        'fd_pct':            0,
        'crypto_pct':       10,
    },
}

ASSET_COLS = ['stocks_pct', 'mutual_funds_pct', 'gold_pct', 'fd_pct', 'crypto_pct']

# Reproducibility
RANDOM_STATE = 42

print('\u2705 Libraries loaded successfully.')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')
print(f'   seaborn : {sns.__version__}')
print()
print('Portfolio Allocation Rules:')
for profile, alloc in PORTFOLIO_RULES.items():
    total = sum(alloc.values())
    print(f'  {profile:<15s}: {alloc}  -> Total={total}%')

✅ Libraries loaded successfully.
   pandas  : 2.3.3
   numpy   : 1.23.5
   seaborn : 0.13.2

Portfolio Allocation Rules:
  Conservative   : {'stocks_pct': 10, 'mutual_funds_pct': 30, 'gold_pct': 20, 'fd_pct': 40, 'crypto_pct': 0}  -> Total=100%
  Moderate       : {'stocks_pct': 30, 'mutual_funds_pct': 40, 'gold_pct': 15, 'fd_pct': 15, 'crypto_pct': 0}  -> Total=100%
  Aggressive     : {'stocks_pct': 50, 'mutual_funds_pct': 30, 'gold_pct': 10, 'fd_pct': 0, 'crypto_pct': 10}  -> Total=100%


---

# Section 3 — Load Datasets

---

## 📎 Dataset Overview

| Dataset | Source | Scope | Key Fields |
|---------|--------|-------|------------|
| `health_score_dataset.csv` | Notebook 04 | All 300 rows (100 accounts × 3 months) | `health_score`, `health_category` |
| `risk_profile_predictions.csv` | Notebook 03 | 60 test-set rows | `predicted_risk_profile`, `prediction_confidence` |

> **Important:** The risk predictions file covers only the 20% test split. Both files contain `account_number` and `month_year`, enabling a precise merge.

In [ ]:
# =============================================================================
# SECTION 3: LOAD DATASETS
# =============================================================================

def load_csv_with_fallback(filename: str) -> pd.DataFrame:
    """
    Load a CSV file from the notebook directory or data/processed/ subdirectory.

    Parameters
    ----------
    filename : str  Name of the CSV file to load.

    Returns
    -------
    pd.DataFrame

    Raises
    ------
    FileNotFoundError  If the file is not found in any search path.
    """
    notebook_dir = os.path.dirname(os.path.abspath('__file__'))
    search_paths = [
        notebook_dir,
        os.path.join(notebook_dir, 'data', 'processed'),
    ]
    for directory in search_paths:
        filepath = os.path.join(directory, filename)
        if os.path.exists(filepath):
            print(f'  \ud83d\udcc2 Loaded : {filepath}')
            return pd.read_csv(filepath, low_memory=False)

    raise FileNotFoundError(
        f"'{filename}' not found in:\n"
        + '\n'.join(f'  - {d}' for d in search_paths)
    )


# ── Load Health Score Dataset (Notebook 04 output) ───────────────────────────
print('\ud83d\udce5 Loading datasets...\n')
health_df = load_csv_with_fallback('health_score_dataset.csv')
risk_df   = load_csv_with_fallback('risk_profile_predictions.csv')

# ── Normalise key join columns to string ──────────────────────────────────────
for df_ in [health_df, risk_df]:
    df_['account_number'] = df_['account_number'].astype(str).str.strip()
    df_['month_year']     = df_['month_year'].astype(str).str.strip()

print()
print('\u2705 Both datasets loaded.')
print(f'   health_score_dataset      : {health_df.shape[0]:,} rows x {health_df.shape[1]} cols')
print(f'   risk_profile_predictions  : {risk_df.shape[0]:,} rows x {risk_df.shape[1]} cols')

In [ ]:
# ── Display sample rows from both datasets ────────────────────────────────────
print('=' * 65)
print('  health_score_dataset.csv — Sample (5 rows)')
print('=' * 65)
print('Columns:', list(health_df.columns))
display(health_df.head())

print()
print('=' * 65)
print('  risk_profile_predictions.csv — Sample (5 rows)')
print('=' * 65)
print('Columns:', list(risk_df.columns))
display(risk_df.head())

In [ ]:
# ── DataFrame Info ────────────────────────────────────────────────────────────
print('health_score_dataset — dtypes:')
health_df.info()
print()
print('risk_profile_predictions — dtypes:')
risk_df.info()

---

# Section 4 — Data Validation & Merge Strategy

---

## 🔄 Merge Strategy

Both datasets contain `account_number` and `month_year`, which together form a composite key.

**Merge type: `inner` join** on `[account_number, month_year]`

This yields rows where **both** a health score **and** a risk prediction exist.

| Scenario | Handling |
|----------|----------|
| Row in health file only | Excluded from merged set (no risk profile available) |
| Row in risk file only | Excluded (no health score available) |
| Row in both files | Included in portfolio recommendation dataset |

> **Assumption:** The 60-row risk prediction file covers the test split (20% of accounts). The merged dataset represents accounts for which both scores are available.

In [ ]:
# =============================================================================
# SECTION 4: DATA VALIDATION & MERGE
# =============================================================================

# ── 4.1 Validate health_df ────────────────────────────────────────────────────
print('=== Validation: health_score_dataset ===')
missing_h = health_df.isnull().sum()
missing_h = missing_h[missing_h > 0]
if missing_h.empty:
    print('  \u2705 No missing values.')
else:
    print(f'  \u26a0\ufe0f  Missing in {len(missing_h)} column(s):')
    display(missing_h.to_frame('Missing Count'))

dup_h = health_df.duplicated(subset=['account_number', 'month_year']).sum()
print(f'  Duplicate (account+month) rows: {dup_h}')

print()

# ── 4.2 Validate risk_df ─────────────────────────────────────────────────────
print('=== Validation: risk_profile_predictions ===')
missing_r = risk_df.isnull().sum()
missing_r = missing_r[missing_r > 0]
if missing_r.empty:
    print('  \u2705 No missing values.')
else:
    print(f'  \u26a0\ufe0f  Missing in {len(missing_r)} column(s):')
    display(missing_r.to_frame('Missing Count'))

dup_r = risk_df.duplicated(subset=['account_number', 'month_year']).sum()
print(f'  Duplicate (account+month) rows: {dup_r}')

print()

# ── 4.3 Key overlap analysis ──────────────────────────────────────────────────
print('=== Merge Key Overlap Analysis ===')
health_keys = set(
    zip(health_df['account_number'], health_df['month_year'])
)
risk_keys = set(
    zip(risk_df['account_number'], risk_df['month_year'])
)

shared_keys      = health_keys & risk_keys
health_only_keys = health_keys - risk_keys
risk_only_keys   = risk_keys   - health_keys

print(f'  Health dataset rows     : {len(health_df):>5,}')
print(f'  Risk dataset rows       : {len(risk_df):>5,}')
print(f'  Shared (account+month)  : {len(shared_keys):>5,}  <- inner join size')
print(f'  Health only             : {len(health_only_keys):>5,}')
print(f'  Risk only               : {len(risk_only_keys):>5,}')

# ── 4.4 Risk profile distribution ────────────────────────────────────────────
print()
print('=== Risk Profile Distribution (risk_profile_predictions) ===')
rp_dist = risk_df['predicted_risk_profile'].value_counts()
for profile, count in rp_dist.items():
    pct = count / len(risk_df) * 100
    print(f'  {profile:<15s}: {count:>4,} ({pct:.1f}%)')

print()
print('=== Health Category Distribution (health_score_dataset) ===')
hc_dist = health_df['health_category'].value_counts()
for cat, count in hc_dist.items():
    pct = count / len(health_df) * 100
    print(f'  {cat:<12s}: {count:>4,} ({pct:.1f}%)')

In [ ]:
# ── 4.5 Perform merge ────────────────────────────────────────────────────────

# Select only the columns needed from the risk predictions file
RISK_COLS_NEEDED = [
    'account_number', 'month_year',
    'predicted_risk_profile',
    'prediction_confidence',
    'actual_risk_profile',
    'correct_prediction',
    'prob_Aggressive', 'prob_Conservative', 'prob_Moderate',
]
risk_cols_present = [c for c in RISK_COLS_NEEDED if c in risk_df.columns]
risk_slim = risk_df[risk_cols_present].copy()

# Inner merge — only rows with both health score AND risk prediction
merged_df = health_df.merge(
    risk_slim,
    on=['account_number', 'month_year'],
    how='inner',
)

print(f'\u2705 Merge complete.')
print(f'   Merged dataset shape : {merged_df.shape[0]:,} rows x {merged_df.shape[1]} columns')
print(f'   Unique accounts      : {merged_df["account_number"].nunique()}')
print(f'   Unique months        : {merged_df["month_year"].nunique()}')
print()
print('  Risk profile counts in merged dataset:')
for profile, count in merged_df['predicted_risk_profile'].value_counts().items():
    print(f'    {profile:<15s}: {count}')
print()
print('  Health category counts in merged dataset:')
for cat, count in merged_df['health_category'].value_counts().items():
    print(f'    {cat:<12s}: {count}')

In [ ]:
# ── 4.6 Summary statistics for key numeric columns ───────────────────────────
print('\ud83d\udcca Summary Statistics — Key Metrics in Merged Dataset\n')
key_numeric = [
    'health_score', 'prediction_confidence',
    'savings_rate_score', 'income_consistency_score_n', 'expense_stability_score_n'
]
available_numeric = [c for c in key_numeric if c in merged_df.columns]
summary = merged_df[available_numeric].describe().round(4).T
display(
    summary.style
    .background_gradient(cmap='Blues', subset=['mean', 'std'])
    .set_caption('Key Metric Statistics — Merged Portfolio Dataset')
)

---

# Section 5 — Portfolio Recommendation Engine

---

The core engine maps each `predicted_risk_profile` to a fixed asset allocation. The allocation dictionary (`PORTFOLIO_RULES`) was defined in Section 2 as a **single source of truth**, ensuring that any future changes propagate automatically throughout the notebook.

In [ ]:
# =============================================================================
# SECTION 5: PORTFOLIO RECOMMENDATION ENGINE
# =============================================================================

def recommend_portfolio(risk_profile: str) -> dict:
    """
    Return the asset allocation dictionary for a given risk profile.

    Allocation percentages are sourced from PORTFOLIO_RULES, ensuring
    a single source of truth across the notebook and any importing module.

    Parameters
    ----------
    risk_profile : str
        One of 'Conservative', 'Moderate', 'Aggressive'.
        Case-insensitive; leading/trailing whitespace is stripped.

    Returns
    -------
    dict
        Keys: 'stocks_pct', 'mutual_funds_pct', 'gold_pct',
              'fd_pct', 'crypto_pct'
        Values: integer percentage allocation (sum = 100).

    Raises
    ------
    ValueError
        If risk_profile is not a recognised label.

    Examples
    --------
    >>> recommend_portfolio('Aggressive')
    {'stocks_pct': 50, 'mutual_funds_pct': 30, 'gold_pct': 10,
     'fd_pct': 0, 'crypto_pct': 10}
    """
    # Normalise input to handle minor formatting issues
    profile_clean = str(risk_profile).strip().capitalize()

    if profile_clean not in PORTFOLIO_RULES:
        valid = list(PORTFOLIO_RULES.keys())
        raise ValueError(
            f"Unknown risk_profile '{risk_profile}'. "
            f"Valid options: {valid}"
        )

    return PORTFOLIO_RULES[profile_clean].copy()


# ── Self-test: verify all profiles return correct totals ─────────────────────
print('\ud83d\udcbc Portfolio Recommendation Engine — Self-Test')
print('=' * 60)

for profile in ['Conservative', 'Moderate', 'Aggressive']:
    allocation = recommend_portfolio(profile)
    total      = sum(allocation.values())
    status     = '\u2705' if total == 100 else '\u274c'
    print(f'  {status} {profile:<15s}: {allocation}  -> Total = {total}%')

# ── Test edge case: case insensitivity ────────────────────────────────────────
try:
    rec = recommend_portfolio('aggressive')   # lowercase input
    print(f'  \u2705 Case normalisation test passed  : {rec}')
except ValueError as e:
    print(f'  \u274c Case normalisation test failed : {e}')

# ── Test edge case: invalid profile ──────────────────────────────────────────
try:
    recommend_portfolio('Unknown')
    print('  \u274c Error handling test FAILED: no exception raised!')
except ValueError as e:
    print(f'  \u2705 Error handling test passed     : {e}')

---

# Section 6 — Generate Portfolio Allocations

---

We apply `recommend_portfolio()` to every row in the merged dataset and validate that all allocations sum to exactly 100%.

In [ ]:
# =============================================================================
# SECTION 6: GENERATE PORTFOLIO ALLOCATIONS
# =============================================================================

def apply_portfolio_allocation(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Apply the portfolio recommendation engine to all rows of the dataframe.

    Creates five new columns: stocks_pct, mutual_funds_pct, gold_pct,
    fd_pct, crypto_pct. Values are sourced from PORTFOLIO_RULES via
    recommend_portfolio().

    Parameters
    ----------
    dataframe : pd.DataFrame
        Must contain a 'predicted_risk_profile' column.

    Returns
    -------
    pd.DataFrame  Dataframe with allocation columns added in-place.
    """
    df_out = dataframe.copy()

    # Vectorised allocation via map (O(n) instead of per-row apply)
    for asset_col in ASSET_COLS:
        df_out[asset_col] = df_out['predicted_risk_profile'].map(
            {profile: alloc[asset_col] for profile, alloc in PORTFOLIO_RULES.items()}
        )

    return df_out


# ── Apply allocations ─────────────────────────────────────────────────────────
reco_df = apply_portfolio_allocation(merged_df)

# ── Validate: all rows must sum to exactly 100 ────────────────────────────────
reco_df['allocation_total'] = reco_df[ASSET_COLS].sum(axis=1)
invalid_rows = reco_df[reco_df['allocation_total'] != 100]

print('\u2705 Portfolio allocations generated.')
print(f'   Records processed   : {len(reco_df):,}')
print(f'   Invalid total rows  : {len(invalid_rows)}')
if not invalid_rows.empty:
    print('   WARNING: The following rows have incorrect totals:')
    display(invalid_rows[['account_number', 'predicted_risk_profile', 'allocation_total'] + ASSET_COLS])
else:
    print('   \u2705 All allocations verified: sum = 100% for every row.')

# ── Sample: show allocation per profile ──────────────────────────────────────
print()
print('Sample allocations by Risk Profile:')
print('-' * 70)
for profile in ['Conservative', 'Moderate', 'Aggressive']:
    sample_rows = reco_df.loc[
        reco_df['predicted_risk_profile'] == profile, ASSET_COLS
    ]
    if not sample_rows.empty:
        row = sample_rows.iloc[0]
        print(
            f'  {profile:<15s}: '
            f'Stocks {int(row.stocks_pct)}%  '
            f'MF {int(row.mutual_funds_pct)}%  '
            f'Gold {int(row.gold_pct)}%  '
            f'FD {int(row.fd_pct)}%  '
            f'Crypto {int(row.crypto_pct)}%'
        )

In [ ]:
# ── Display enriched dataset sample ──────────────────────────────────────────
PREVIEW_COLS = [
    'account_number', 'month_year', 'predicted_risk_profile',
    'prediction_confidence', 'health_score', 'health_category',
] + ASSET_COLS + ['allocation_total']

print('\ud83d\udcc4 Merged + Allocated Dataset Preview (10 rows):')
display(
    reco_df[PREVIEW_COLS].head(10).style
    .background_gradient(cmap='RdYlGn', subset=['health_score'])
    .applymap(
        lambda v: f'color: {RISK_COLORS.get(str(v), "black")}; font-weight: bold',
        subset=['predicted_risk_profile']
    )
)

---

# Section 7 — Investment Strategy Generator

---

Each risk profile receives a concise **investment philosophy statement** that:
- Communicates the core objective behind the allocation
- Is displayed on the WealthWise AI dashboard as context
- Provides the AI Coach with a strategy framing for the conversation

In [ ]:
# =============================================================================
# SECTION 7: INVESTMENT STRATEGY GENERATOR
# =============================================================================

# Strategy templates — keyed by risk profile
INVESTMENT_STRATEGIES = {
    'Conservative': (
        "Focus on capital preservation and stable long-term growth. "
        "Your portfolio prioritises Fixed Deposits and Mutual Funds to protect "
        "principal while generating steady, low-risk returns. "
        "Gold allocation provides a reliable inflation hedge."
    ),
    'Moderate': (
        "Balance growth opportunities with portfolio stability. "
        "A diversified mix of Mutual Funds and Stocks enables participation in "
        "market growth, while Gold and Fixed Deposits provide a stability cushion "
        "during market downturns."
    ),
    'Aggressive': (
        "Focus on long-term wealth creation while accepting higher short-term "
        "volatility. A dominant equity position combined with Crypto exposure targets "
        "maximum capital appreciation over a 5-10 year investment horizon. "
        "Suitable for investors with a strong income base and long time horizon."
    ),
}


def generate_investment_strategy(risk_profile: str) -> str:
    """
    Return the investment strategy text for a given risk profile.

    Parameters
    ----------
    risk_profile : str  One of 'Conservative', 'Moderate', 'Aggressive'.

    Returns
    -------
    str  Strategy description.
    """
    profile_clean = str(risk_profile).strip().capitalize()
    return INVESTMENT_STRATEGIES.get(
        profile_clean,
        'Strategy not available. Please verify risk profile classification.'
    )


# ── Apply to merged dataset ───────────────────────────────────────────────────
reco_df['investment_strategy'] = reco_df['predicted_risk_profile'].map(
    INVESTMENT_STRATEGIES
)

print('\u2705 Investment strategies assigned.')
print(f'   Null strategies: {reco_df["investment_strategy"].isnull().sum()}')
print()

# ── Display one strategy per profile ─────────────────────────────────────────
print('Investment Strategies by Profile:')
print('=' * 70)
for profile, strategy in INVESTMENT_STRATEGIES.items():
    count = (reco_df['predicted_risk_profile'] == profile).sum()
    print(f'\n  [{profile}]  ({count} records)')
    # Word-wrap at 80 chars
    import textwrap
    for line in textwrap.wrap(strategy, width=75):
        print(f'    {line}')

---

# Section 8 — Health Score Based Advice

---

The `investment_advice` field layers the **Financial Health Score** on top of the risk-profile strategy. While the investment strategy tells *where* to invest, the advice tells the user *whether they are ready* to invest at their intended risk level.

| Health Score | Advice Tier |
|-------------|-------------|
| ≥ 80 | 🟢 Excellent — ready to maximise investments |
| 60–79 | 🟡 Good — maintain discipline and invest steadily |
| 40–59 | 🟠 Fair — stabilise before increasing risk exposure |
| < 40 | 🔴 Poor — address fundamentals before any aggressive investing |

In [ ]:
# =============================================================================
# SECTION 8: HEALTH SCORE BASED INVESTMENT ADVICE
# =============================================================================

def generate_investment_advice(
    health_score: float,
    risk_profile: str,
) -> str:
    """
    Generate personalised investment advice combining health score
    and risk profile.

    The advice is calibrated to the user's current financial readiness
    and contextualised by their risk classification. Users with poor
    health scores are advised to stabilise finances before investing
    at their risk profile's full capacity.

    Parameters
    ----------
    health_score : float  Financial Health Score [0, 100].
    risk_profile : str    'Conservative', 'Moderate', or 'Aggressive'.

    Returns
    -------
    str  Personalised, actionable investment advice.
    """
    profile = str(risk_profile).strip().capitalize()

    # ── Excellent health (>= 80) ─────────────────────────────────────────────
    if health_score >= 80:
        if profile == 'Conservative':
            return (
                'Excellent financial health. Your stable profile is ideal for "
                'compounding through long-term FD laddering and debt Mutual Funds. "
                'Consider allocating your annual bonus to a Sovereign Gold Bond for '
                'additional tax-efficient returns.'
            )
        if profile == 'Moderate':
            return (
                'Excellent financial health. Consider increasing your Mutual Fund SIP "
                'by 10-15% and explore index fund investing to participate in "
                'long-term market growth while maintaining your balanced allocation.'
            )
        # Aggressive
        return (
            'Excellent financial health. Your strong savings base supports your "
            'aggressive allocation. Maximise your ELSS (tax-saving equity fund) "
            'contributions and maintain a 6-month emergency fund before increasing "
            'Crypto exposure.'
        )

    # ── Good health (60-79) ──────────────────────────────────────────────────
    if health_score >= 60:
        if profile == 'Conservative':
            return (
                'Good financial health. Maintain your current savings discipline. "
                'Ensure all FDs are auto-renewed and consider laddering maturities "
                'across 1, 2, and 3-year tenures for optimal liquidity.'
            )
        if profile == 'Moderate':
            return (
                'Good financial health. Maintain current savings discipline and "
                'start a monthly SIP in a diversified equity Mutual Fund. "
                'Gradually increase equity exposure as income grows.'
            )
        # Aggressive
        return (
            'Good financial health. Continue building savings while pursuing your "
            'growth strategy. Limit Crypto to the recommended 10% allocation and "
            'focus the majority of equity exposure on diversified index funds.'
        )

    # ── Fair health (40-59) ──────────────────────────────────────────────────
    if health_score >= 40:
        if profile == 'Aggressive':
            return (
                'Moderate financial health. Improve savings rate and reduce expense "
                'variability before maintaining full Aggressive exposure. "
                'Consider temporarily shifting 10% from Stocks to Fixed Deposit "
                'until your health score reaches 60.'
            )
        if profile == 'Moderate':
            return (
                'Moderate financial health. Focus on expense reduction before "
                'increasing equity exposure. Prioritise building a 3-month "
                'emergency fund and a recurring FD before increasing SIP amounts.'
            )
        # Conservative
        return (
            'Moderate financial health. Your Conservative allocation is appropriate. "
            'Focus on improving savings rate consistency before considering any "
            'shift toward higher-risk asset classes.'
        )

    # ── Poor health (< 40) ───────────────────────────────────────────────────
    if profile == 'Aggressive':
        return (
            'Your financial health score is in the Poor range. Before pursuing an "
            'Aggressive investment strategy, prioritise: (1) eliminating overspending, "
            '(2) building a minimum 2-month emergency fund, and "
            '(3) achieving a positive monthly savings rate. Revisit investment "
            'allocation once health score reaches 40+.'
        )
    if profile == 'Moderate':
        return (
            'Your financial health score is in the Poor range. Focus on reducing "
            'expenses and building emergency savings before increasing risk exposure. "
            'Start with a small recurring FD and build savings consistency over "
            '3 months before reviewing your portfolio allocation.'
        )
    # Conservative
    return (
        'Your financial health score is in the Poor range. Your Conservative "
        'allocation is the safest path for now. Focus on reducing expenses and "
        'building emergency savings. Consult the WealthWise AI Financial Coach "
        'for a personalised recovery plan.'
    )


# ── Apply to merged dataset ───────────────────────────────────────────────────
reco_df['investment_advice'] = reco_df.apply(
    lambda row: generate_investment_advice(
        health_score = row['health_score'],
        risk_profile = row['predicted_risk_profile'],
    ),
    axis=1,
)

print('\u2705 Investment advice generated for all', len(reco_df), 'records.')
print(f'   Null advice: {reco_df["investment_advice"].isnull().sum()}')
print(f'   Unique advice variants: {reco_df["investment_advice"].nunique()}')

In [ ]:
# ── Sample advice display across all combinations ─────────────────────────────
print('\ud83d\udcac Sample Personalised Investment Advice')
print('=' * 70)

shown = set()
for _, row in reco_df.sort_values('health_score', ascending=False).iterrows():
    key = (row['predicted_risk_profile'], row['health_category'])
    if key in shown:
        continue
    shown.add(key)
    print(
        f"\n  Profile: {row['predicted_risk_profile']:<15s}  "
        f"Health: {row['health_category']:<10s}  "
        f"Score: {row['health_score']:.1f}/100"
    )
    import textwrap
    for line in textwrap.wrap(row['investment_advice'], width=72):
        print(f'    {line}')

print(f'\n  Total unique combinations shown: {len(shown)}')

---

# Section 9 — Visualisation

---

Four targeted charts that communicate the portfolio recommendation outputs clearly to both technical and business audiences.

In [ ]:
# =============================================================================
# SECTION 9: VISUALISATION
# =============================================================================

# ── 9.1 Risk Profile Distribution ────────────────────────────────────────────
rp_order   = [p for p in ['Conservative', 'Moderate', 'Aggressive']
              if p in reco_df['predicted_risk_profile'].unique()]
rp_counts  = reco_df['predicted_risk_profile'].value_counts()
rp_pcts    = reco_df['predicted_risk_profile'].value_counts(normalize=True).mul(100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=FIGURE_DPI)
fig.suptitle(
    'WealthWise AI \u2014 Risk Profile Distribution (Merged Dataset)',
    fontsize=14, fontweight='bold', y=1.02
)

# Bar
ax1 = axes[0]
bars = ax1.bar(
    rp_order,
    [rp_counts.get(p, 0) for p in rp_order],
    color=[RISK_COLORS[p] for p in rp_order],
    edgecolor='white', linewidth=1.5, width=0.5
)
for bar, p in zip(bars, rp_order):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f'{rp_counts.get(p,0):,}\n({rp_pcts.get(p,0):.1f}%)',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )
ax1.set_title('Count by Risk Profile', fontsize=12, fontweight='bold')
ax1.set_xlabel('Risk Profile', fontsize=11)
ax1.set_ylabel('Number of Records', fontsize=11)
ax1.spines[['top', 'right']].set_visible(False)

# Pie
ax2 = axes[1]
pie_vals = [rp_counts.get(p, 0) for p in rp_order]
pie_cols = [RISK_COLORS[p] for p in rp_order]
wedges, _, autotexts = ax2.pie(
    pie_vals, labels=None, colors=pie_cols,
    autopct='%1.1f%%', startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    pctdistance=0.78,
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
    at.set_color('white')
# Donut
ax2.add_patch(plt.Circle((0, 0), 0.50, color='white'))
patches = [mpatches.Patch(color=RISK_COLORS[p], label=p) for p in rp_order]
ax2.legend(handles=patches, loc='lower center',
           bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=10)
ax2.set_title('Portfolio — Risk Profile Split', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('risk_profile_distribution_reco.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('\ud83d\udcca Chart 1 saved: risk_profile_distribution_reco.png')

# ── Markdown Interpretation ───────────────────────────────────────────────────
dominant = max(rp_order, key=lambda p: rp_counts.get(p, 0))
print(f"\n\u2139\ufe0f  Interpretation: '{dominant}' is the dominant risk profile "
      f"({rp_pcts.get(dominant,0):.1f}% of merged records). "
      "This is expected given the dataset's savings rate distribution.")

In [ ]:
# ── 9.2 Health Category Distribution ─────────────────────────────────────────
hc_order  = [c for c in ['Poor', 'Fair', 'Good', 'Excellent']
             if c in reco_df['health_category'].unique()]
hc_counts = reco_df['health_category'].value_counts()
hc_pcts   = reco_df['health_category'].value_counts(normalize=True).mul(100)

fig, ax = plt.subplots(figsize=(10, 5), dpi=FIGURE_DPI)

bars = ax.barh(
    hc_order,
    [hc_counts.get(c, 0) for c in hc_order],
    color=[HEALTH_COLORS[c] for c in hc_order],
    edgecolor='white', linewidth=1.2, height=0.5,
)
for bar, cat in zip(bars, hc_order):
    ax.text(
        bar.get_width() + 0.3,
        bar.get_y() + bar.get_height() / 2,
        f'{hc_counts.get(cat,0):,}  ({hc_pcts.get(cat,0):.1f}%)',
        va='center', ha='left', fontsize=11, fontweight='bold'
    )
ax.set_title(
    'Health Category Distribution \u2014 Merged Dataset',
    fontsize=13, fontweight='bold', pad=12
)
ax.set_xlabel('Number of Records', fontsize=11)
ax.set_ylabel('Health Category', fontsize=11)
ax.set_xlim(0, max(hc_counts.values) * 1.25)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('health_category_distribution_reco.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('\ud83d\udcca Chart 2 saved: health_category_distribution_reco.png')
print()
print('\u2139\ufe0f  Interpretation: Most users in the merged dataset are in the Poor/Fair '
      'health category, indicating significant opportunity for coaching and financial improvement.')

In [ ]:
# ── 9.3 Portfolio Allocation Comparison (Grouped Bar) ─────────────────────────
profiles_present = [p for p in ['Conservative', 'Moderate', 'Aggressive']
                    if p in reco_df['predicted_risk_profile'].unique()]

asset_labels   = ['Stocks', 'Mutual Funds', 'Gold', 'Fixed Deposit', 'Crypto']
asset_col_map  = dict(zip(asset_labels, ASSET_COLS))

x         = np.arange(len(asset_labels))
n_profiles = len(profiles_present)
bar_w      = 0.25

fig, ax = plt.subplots(figsize=(14, 6), dpi=FIGURE_DPI)

for i, profile in enumerate(profiles_present):
    values = [
        PORTFOLIO_RULES[profile][asset_col_map[label]]
        for label in asset_labels
    ]
    offset = (i - (n_profiles - 1) / 2) * (bar_w + 0.02)
    bars = ax.bar(
        x + offset, values, bar_w,
        label=profile,
        color=RISK_COLORS[profile],
        edgecolor='white', linewidth=1, alpha=0.88
    )
    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.4,
                f'{val}%',
                ha='center', va='bottom', fontsize=9, fontweight='bold'
            )

ax.set_xticks(x)
ax.set_xticklabels(asset_labels, fontsize=12)
ax.set_ylabel('Allocation (%)', fontsize=11)
ax.set_ylim(0, 62)
ax.set_title(
    'Portfolio Allocation Comparison by Risk Profile',
    fontsize=14, fontweight='bold', pad=14
)
ax.legend(fontsize=11, loc='upper right')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('portfolio_allocation_comparison.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('\ud83d\udcca Chart 3 saved: portfolio_allocation_comparison.png')
print()
print('\u2139\ufe0f  Interpretation: As risk profile moves from Conservative to Aggressive, '
      'equity exposure (Stocks + Crypto) increases while capital preservation '
      'instruments (FD) decrease. Mutual Funds remain significant across all profiles '
      'due to their diversification benefits.')

In [ ]:
# ── 9.4 Asset Allocation by Risk Profile (Stacked Pie Grid) ──────────────────
profiles_for_pie = [p for p in ['Conservative', 'Moderate', 'Aggressive']
                    if p in PORTFOLIO_RULES]

fig, axes = plt.subplots(1, len(profiles_for_pie),
                         figsize=(16, 6), dpi=FIGURE_DPI)
fig.suptitle(
    'Asset Allocation Breakdown by Risk Profile',
    fontsize=14, fontweight='bold', y=1.02
)

if len(profiles_for_pie) == 1:
    axes = [axes]

for ax, profile in zip(axes, profiles_for_pie):
    alloc  = PORTFOLIO_RULES[profile]
    labels = [a for a, v in zip(asset_labels, ASSET_COLS) if alloc[v] > 0]
    values = [alloc[v] for v in ASSET_COLS if alloc[v] > 0]
    colors = [ASSET_COLORS[a] for a in labels]

    wedges, texts, autotexts = ax.pie(
        values, labels=None, colors=colors,
        autopct='%1.0f%%', startangle=140,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2},
        pctdistance=0.75,
    )
    for at in autotexts:
        at.set_fontsize(11)
        at.set_fontweight('bold')
        at.set_color('white')

    # Donut + profile label
    ax.add_patch(plt.Circle((0, 0), 0.48, color='white'))
    ax.text(0, 0, profile, ha='center', va='center',
            fontsize=11, fontweight='bold', color=RISK_COLORS[profile])

    # Legend
    legend_patches = [
        mpatches.Patch(color=ASSET_COLORS[l], label=f'{l}: {v}%')
        for l, v in zip(labels, values)
    ]
    ax.legend(
        handles=legend_patches,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.22),
        ncol=2, fontsize=9,
        framealpha=0.9,
    )
    ax.set_title(f'{profile} Portfolio', fontsize=12,
                 fontweight='bold', pad=14, color=RISK_COLORS[profile])

plt.tight_layout()
plt.savefig('asset_allocation_by_profile.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('\ud83d\udcca Chart 4 saved: asset_allocation_by_profile.png')
print()
print('\u2139\ufe0f  Interpretation: Each donut chart clearly visualises the asset mix for a "
      "single risk profile. Conservative portfolios are FD-heavy; Aggressive "
      "portfolios are equity-heavy. All three pie charts sum to 100%.')

---

# Section 10 — Recommendation Summary

---

We display the **Top 10 highest health score users** from the merged dataset along with their complete portfolio recommendation.

In [ ]:
# =============================================================================
# SECTION 10: RECOMMENDATION SUMMARY
# =============================================================================

SUMMARY_COLS = [
    'account_number', 'month_year',
    'health_score', 'health_category',
    'predicted_risk_profile', 'prediction_confidence',
] + ASSET_COLS + ['investment_strategy', 'investment_advice']

available_summary_cols = [c for c in SUMMARY_COLS if c in reco_df.columns]

top_10 = (
    reco_df
    .nlargest(10, 'health_score')
    [available_summary_cols]
    .reset_index(drop=True)
)
top_10.index = top_10.index + 1   # 1-based rank

print('\ud83c\udfc6 Top 10 Highest Health Score Users — Portfolio Recommendations')
print('=' * 70)

# Display compact view (exclude long text columns for readability)
compact_cols = [
    'account_number', 'month_year',
    'health_score', 'health_category',
    'predicted_risk_profile', 'prediction_confidence',
] + ASSET_COLS
compact_cols = [c for c in compact_cols if c in top_10.columns]

display(
    top_10[compact_cols].style
    .background_gradient(cmap='Greens', subset=['health_score'])
    .background_gradient(cmap='Blues',  subset=['prediction_confidence'])
    .set_caption('Top 10 Accounts by Financial Health Score')
)

# Full detail for top 3
print()
print('\ud83d\udd0d Detailed View \u2014 Top 3 Accounts:')
print('=' * 70)
for rank, row in top_10.head(3).iterrows():
    print(f"\n  Rank {rank}  |  Account: {row['account_number']}  "
          f"|  Month: {row['month_year']}")
    print(f"  Health Score      : {row['health_score']:.2f} / 100  "
          f"[{row['health_category']}]")
    print(f"  Risk Profile      : {row['predicted_risk_profile']}  "
          f"(confidence: {row['prediction_confidence']:.2%})")
    print(f"  Allocation        : "
          f"Stocks {int(row.get('stocks_pct',0))}%  "
          f"MF {int(row.get('mutual_funds_pct',0))}%  "
          f"Gold {int(row.get('gold_pct',0))}%  "
          f"FD {int(row.get('fd_pct',0))}%  "
          f"Crypto {int(row.get('crypto_pct',0))}%")
    if 'investment_advice' in row.index:
        import textwrap
        for line in textwrap.wrap(str(row['investment_advice']), width=68):
            print(f'  Advice            : {line}')

In [ ]:
# ── Top 10: Visual Bar Chart ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6), dpi=FIGURE_DPI)

y_labels = [
    f"{row['account_number']} | {row['month_year']} | {row['predicted_risk_profile']}"
    for _, row in top_10.iterrows()
]
scores = top_10['health_score'].values
bar_colors = [
    RISK_COLORS.get(row['predicted_risk_profile'], '#3498DB')
    for _, row in top_10.iterrows()
]

bars = ax.barh(
    range(len(scores)), scores,
    color=bar_colors, alpha=0.85,
    edgecolor='white', linewidth=0.8
)
for bar, score, cat in zip(bars, scores, top_10['health_category']):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f'{score:.1f}  [{cat}]',
        va='center', ha='left', fontsize=10, fontweight='bold'
    )

ax.set_yticks(range(len(y_labels)))
ax.set_yticklabels(y_labels, fontsize=9)
ax.invert_yaxis()
ax.set_xlim(0, 115)
ax.set_xlabel('Health Score (0\u2013100)', fontsize=11)
ax.set_title(
    'Top 10 Accounts by Health Score \u2014 Colour-Coded by Risk Profile',
    fontsize=13, fontweight='bold', pad=14
)

legend_patches = [mpatches.Patch(color=c, label=p)
                  for p, c in RISK_COLORS.items()]
ax.legend(handles=legend_patches, fontsize=10, loc='lower right')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('top10_recommendations.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('\ud83d\udcca Chart 5 saved: top10_recommendations.png')

---

# Section 11 — Export Results

---

The final export includes all columns required for the FastAPI backend, WealthWise AI dashboard, and AI Financial Coach integration.

In [ ]:
# =============================================================================
# SECTION 11: EXPORT RESULTS
# =============================================================================

# ── Define export column order ────────────────────────────────────────────────
EXPORT_COLS_ORDERED = [
    # Identifiers
    'account_number',
    'month_year',

    # Account metadata (if present)
    'account_holder',
    'bank_name',
    'account_type',

    # Risk Profile
    'predicted_risk_profile',
    'prediction_confidence',
    'actual_risk_profile',
    'correct_prediction',
    'prob_Aggressive',
    'prob_Conservative',
    'prob_Moderate',

    # Financial Health
    'health_score',
    'health_category',
    'strengths',
    'weaknesses',
    'health_insight',

    # Portfolio Allocation
    'stocks_pct',
    'mutual_funds_pct',
    'gold_pct',
    'fd_pct',
    'crypto_pct',

    # Advice & Strategy
    'investment_strategy',
    'investment_advice',

    # Key financial metrics
    'monthly_income',
    'monthly_expense',
    'savings',
    'savings_rate',
    'average_balance',
    'transaction_count',
    'savings_rate_score',
    'income_consistency_score_n',
    'expense_stability_score_n',

    # Alert flags
    'high_spending_flag',
    'high_cash_withdrawal_flag',
    'low_savings_flag',
]

# Keep only columns that exist in the merged dataframe
export_cols = [c for c in EXPORT_COLS_ORDERED if c in reco_df.columns]
export_df   = reco_df[export_cols].copy()

# Drop the working column used for validation
if 'allocation_total' in export_df.columns:
    export_df.drop(columns=['allocation_total'], inplace=True, errors='ignore')

print(f'Export dataset shape : {export_df.shape[0]:,} rows x {export_df.shape[1]} columns')
print(f'Output columns ({len(export_cols)}):')
for i, col in enumerate(export_cols, 1):
    print(f'  {i:02d}. {col}')

In [ ]:
# ── Export: CSV ───────────────────────────────────────────────────────────────
CSV_PATH = 'portfolio_recommendation_dataset.csv'
export_df.to_csv(CSV_PATH, index=False)
csv_size = os.path.getsize(CSV_PATH) / 1024
print(f'\u2705 CSV  exported -> {CSV_PATH}  ({csv_size:.1f} KB)')


# ── Export: Excel (multi-sheet) ───────────────────────────────────────────────
XLSX_PATH = 'portfolio_recommendation_dataset.xlsx'

try:
    with pd.ExcelWriter(XLSX_PATH, engine='openpyxl') as writer:

        # Sheet 1: Full recommendation dataset
        export_df.to_excel(writer, sheet_name='Portfolio_Recommendations', index=False)

        # Sheet 2: Summary by risk profile
        rp_summary = reco_df.groupby('predicted_risk_profile').agg(
            Records              = ('health_score', 'count'),
            Avg_Health_Score     = ('health_score', 'mean'),
            Avg_Confidence       = ('prediction_confidence', 'mean'),
            Stocks_Pct           = ('stocks_pct', 'mean'),
            Mutual_Funds_Pct     = ('mutual_funds_pct', 'mean'),
            Gold_Pct             = ('gold_pct', 'mean'),
            FD_Pct               = ('fd_pct', 'mean'),
            Crypto_Pct           = ('crypto_pct', 'mean'),
        ).round(2)
        rp_summary.to_excel(writer, sheet_name='Summary_by_Risk_Profile')

        # Sheet 3: Summary by health category
        hc_summary = reco_df.groupby('health_category').agg(
            Records              = ('health_score', 'count'),
            Min_Score            = ('health_score', 'min'),
            Mean_Score           = ('health_score', 'mean'),
            Max_Score            = ('health_score', 'max'),
            Aggressive_Count     = ('predicted_risk_profile',
                                    lambda x: (x == 'Aggressive').sum()),
            Moderate_Count       = ('predicted_risk_profile',
                                    lambda x: (x == 'Moderate').sum()),
            Conservative_Count   = ('predicted_risk_profile',
                                    lambda x: (x == 'Conservative').sum()),
        ).round(2).reindex(['Poor', 'Fair', 'Good', 'Excellent'])
        hc_summary.to_excel(writer, sheet_name='Summary_by_Health_Category')

        # Sheet 4: Top 10 accounts
        top_10[compact_cols + ['investment_advice']].to_excel(
            writer, sheet_name='Top_10_Accounts'
        )

    xlsx_size = os.path.getsize(XLSX_PATH) / 1024
    print(f'\u2705 Excel exported -> {XLSX_PATH}  ({xlsx_size:.1f} KB)')
    print('   Sheets: Portfolio_Recommendations | Summary_by_Risk_Profile | '
          'Summary_by_Health_Category | Top_10_Accounts')

except ImportError:
    print('\u26a0\ufe0f  openpyxl not installed. Run: pip install openpyxl')
    print(f'   CSV export available at: {CSV_PATH}')

print()
print('\ud83d\udcce Export dataset preview:')
display(export_df.head(5))

In [ ]:
# ── Export Quality Gate ───────────────────────────────────────────────────────
print('\ud83d\udd0d Export Quality Verification')
print('=' * 60)

quality_checks = [
    ('No missing account_number',
     export_df['account_number'].notna().all()),
    ('No missing predicted_risk_profile',
     export_df['predicted_risk_profile'].notna().all()),
    ('No missing health_score',
     export_df['health_score'].notna().all()),
    ('health_score in [0, 100]',
     export_df['health_score'].between(0, 100).all()),
    ('All allocations sum to 100%',
     (export_df[ASSET_COLS].sum(axis=1) == 100).all()),
    ('investment_strategy not null',
     export_df['investment_strategy'].notna().all()),
    ('investment_advice not null',
     export_df['investment_advice'].notna().all()),
    ('No duplicate (account+month) rows',
     not export_df.duplicated(subset=['account_number', 'month_year']).any()),
]

all_passed = True
for description, result in quality_checks:
    icon = '\u2705' if result else '\u274c'
    print(f'  {icon}  {description}')
    if not result:
        all_passed = False

print()
if all_passed:
    print('  \u2705 ALL QUALITY CHECKS PASSED. Dataset is production-ready.')
else:
    print('  \u274c Some checks failed. Review before deploying.')

---

# Section 12 — Business Interpretation

---

## 🏢 How the Portfolio Recommendation Engine Powers WealthWise AI

### Full Platform Data Flow

```
Raw Bank Statement (PDF / JSON)
         |
         v
[Notebook 01] Data Cleaning
         |
         v
[Notebook 02] Feature Engineering
  -> features_dataset.csv  (37 features per account-month)
         |
         +----------------------------+
         |                            |
         v                            v
[Notebook 03]                [Notebook 04]
Risk Profile Model           Financial Health Score
-> Conservative /            -> Score: 0-100
   Moderate /                -> Category: Poor/Fair/Good/Excellent
   Aggressive                -> Strengths & Weaknesses
         |                            |
         +----------------------------+
                      |
                      v
            [Notebook 05]  <-- YOU ARE HERE
      Portfolio Recommendation Engine
      |
      +-- Risk Profile -> Asset Allocation (Stocks/MF/Gold/FD/Crypto)
      +-- Health Score -> Investment Readiness Advice
      +-- Combined    -> investment_strategy + investment_advice
                      |
                      v
        portfolio_recommendation_dataset.csv / .xlsx
                      |
                      v
            [Notebook 06]  AI Financial Coach
      Uses: risk_profile + health_score + investment_advice
      -> Gemini-powered conversational guidance
      -> Context: "Your Aggressive profile suggests 50% equities..."
```

---

### 🚀 FastAPI Backend Integration Pattern

```python
# app/services/portfolio_service.py

PORTFOLIO_RULES = {
    'Conservative': {'stocks_pct': 10, 'mutual_funds_pct': 30,
                     'gold_pct': 20, 'fd_pct': 40, 'crypto_pct': 0},
    'Moderate':     {'stocks_pct': 30, 'mutual_funds_pct': 40,
                     'gold_pct': 15, 'fd_pct': 15, 'crypto_pct': 0},
    'Aggressive':   {'stocks_pct': 50, 'mutual_funds_pct': 30,
                     'gold_pct': 10, 'fd_pct': 0,  'crypto_pct': 10},
}

class PortfolioRecommendationService:
    """
    WealthWise AI Portfolio Recommendation Service.
    Accepts risk_profile + health_score; returns allocation + advice.
    """

    def recommend(
        self,
        risk_profile: str,
        health_score: float,
    ) -> dict:
        allocation = PORTFOLIO_RULES[risk_profile].copy()
        strategy   = generate_investment_strategy(risk_profile)
        advice     = generate_investment_advice(health_score, risk_profile)

        return {
            'risk_profile':        risk_profile,
            'health_score':        health_score,
            'allocation':          allocation,
            'investment_strategy': strategy,
            'investment_advice':   advice,
        }


# FastAPI usage:
# @router.post('/api/recommend-portfolio')
# async def recommend_portfolio(req: PortfolioRequest):
#     service = PortfolioRecommendationService()
#     return service.recommend(req.risk_profile, req.health_score)
#
# Example response:
# {
#   'risk_profile': 'Moderate',
#   'health_score': 62.4,
#   'allocation': {'stocks_pct': 30, 'mutual_funds_pct': 40, ...},
#   'investment_strategy': 'Balance growth opportunities...',
#   'investment_advice': 'Good financial health. Maintain current...'
# }
```

---

### 📊 How Users Benefit from the Portfolio Recommendation

| User Interaction | Portfolio Recommendation Role |
|-----------------|------------------------------|
| Dashboard landing page | See allocation pie chart with personal percentages |
| Investment tab | Detailed breakdown by asset class with \u20b9 amounts if investable amount entered |
| AI Coach chat | Coach references `investment_strategy` as opening context |
| Monthly review | Allocation recalculates automatically if risk profile changes |
| Health score improvement | Advice updates in real time as health score crosses tier boundaries |

---

### 📊 Extending the Engine

The current engine uses **rule-based fixed allocations** (production-appropriate for a first release). Future versions of WealthWise AI could extend this to:

| Enhancement | Method |
|-------------|--------|
| Dynamic allocation | Modern Portfolio Theory (MPT) optimiser using live market data |
| Goal-based investing | Adjust allocation based on user-stated financial goals |
| Tax optimisation | ELSS weighting for Indian tax-saving under Section 80C |
| Micro-allocation | Sub-allocate within equity to Large/Mid/Small Cap MFs |
| Real-time rebalancing | Trigger rebalance alerts when portfolio drifts > 5% |

---

## ✅ Notebook 05 Summary

| Section | Task | Status |
|---------|------|--------|
| 1  | Platform overview and portfolio theory | ✅ Complete |
| 2  | Library imports + PORTFOLIO_RULES config | ✅ Complete |
| 3  | Load health_score + risk_predictions datasets | ✅ Complete |
| 4  | Validation + inner merge on account+month | ✅ Complete |
| 5  | `recommend_portfolio()` engine with tests | ✅ Complete |
| 6  | Vectorised allocation + 100% sum validation | ✅ Complete |
| 7  | Investment strategy generator | ✅ Complete |
| 8  | Health-score-layered investment advice (12 branches) | ✅ Complete |
| 9  | 5 professional charts | ✅ Complete |
| 10 | Top 10 recommendations with full detail | ✅ Complete |
| 11 | CSV + Excel (4 sheets) + 8-point quality gate | ✅ Complete |
| 12 | Business interpretation + FastAPI pattern | ✅ Complete |

> **Next Step:** Run `06_ai_financial_coach.ipynb` to build the Gemini-powered conversational AI coach that uses risk profile, health score, and portfolio recommendations as context.

In [ ]:
# =============================================================================
# FINAL SUMMARY — Output Verification
# =============================================================================

print('=' * 65)
print('  WealthWise AI -- Portfolio Recommendation: Final Summary')
print('=' * 65)

print('\n  Input Datasets')
print(f'    health_score_dataset    : {len(health_df):,} rows')
print(f'    risk_profile_predictions: {len(risk_df):,} rows')
print(f'    Merged (inner join)     : {len(reco_df):,} rows')

print('\n  Portfolio Allocation Rules Verified')
for profile, alloc in PORTFOLIO_RULES.items():
    total = sum(alloc.values())
    icon  = '\u2705' if total == 100 else '\u274c'
    print(f'    {icon} {profile:<15s}: total = {total}%')

print('\n  Risk Profile Distribution (merged dataset)')
for profile, count in reco_df['predicted_risk_profile'].value_counts().items():
    pct = count / len(reco_df) * 100
    bar = '|' * int(pct / 5)
    print(f'    {profile:<15s}: {count:>4,} ({pct:>5.1f}%)  {bar}')

print('\n  Health Category Distribution (merged dataset)')
for cat in ['Excellent', 'Good', 'Fair', 'Poor']:
    count = (reco_df['health_category'] == cat).sum()
    if count > 0:
        pct = count / len(reco_df) * 100
        print(f'    {cat:<12s}: {count:>4,} ({pct:>5.1f}%)')

print('\n  Output Files')
for fname, desc in [
    ('portfolio_recommendation_dataset.csv',  'Full recommendation dataset (CSV)'),
    ('portfolio_recommendation_dataset.xlsx', 'Full dataset + summaries (Excel, 4 sheets)'),
]:
    exists = os.path.exists(fname)
    icon   = '\u2705' if exists else '\u274c'
    size   = f'{os.path.getsize(fname)/1024:.1f} KB' if exists else 'Not found'
    print(f'    {icon} {fname:<45s} {size:>10s}')

print('\n  Charts Generated')
charts = [
    'risk_profile_distribution_reco.png',
    'health_category_distribution_reco.png',
    'portfolio_allocation_comparison.png',
    'asset_allocation_by_profile.png',
    'top10_recommendations.png',
]
for chart in charts:
    icon = '\u2705' if os.path.exists(chart) else '\u2014'
    print(f'    {icon} {chart}')

print('\n  Next Step: 06_ai_financial_coach.ipynb')
print('=' * 65)